# Teacher-Student Comparison Dashboard

This notebook focuses on the teacher-student pipeline:

- teacher dataset size and chronic coverage,
- teacher action-0 / non-idle balance,
- heuristic overwrite and force-noop rates,
- behavior-cloning training curves stored inside student checkpoints,
- student predicted non-idle rate vs teacher target non-idle rate,
- standalone full-test JSON summaries for student checkpoints.

It complements `policy_comparison_dashboard.ipynb`, which focuses on the RL training runs and W&B metrics.

In [ ]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    local = candidate / "comparison_dashboard.py"
    repo = candidate / "Topology_Task" / "configs" / "zz_print_metrics" / "comparison_dashboard.py"
    if local.exists():
        sys.path.insert(0, str(candidate))
        break
    if repo.exists():
        sys.path.insert(0, str(repo.parent))
        break

import comparison_dashboard as cd
cd = importlib.reload(cd)
print("comparison_dashboard:", cd.__file__)
print("TASK_DIR:", cd.TASK_DIR)

## Configuration

In [ ]:
DATASET_ROOT = cd.TEACHER_DATASET_ROOT
STUDENT_CHECKPOINT_DIR = cd.TEACHER_STUDENT_CHECKPOINT_DIR
SHOW_FIGURES = True
SAVE_FIGURES = True

print("Dataset root:", DATASET_ROOT)
print("Student checkpoint dir:", STUDENT_CHECKPOINT_DIR)

## Teacher Dataset Summaries

In [ ]:
dataset_df, dataset_agent_df = cd.load_teacher_dataset_summaries(DATASET_ROOT)
print("datasets:", len(dataset_df))
display(dataset_df.sort_values("dataset") if not dataset_df.empty else dataset_df)
print("agent summaries:")
display(dataset_agent_df.sort_values(["dataset", "agent"]) if not dataset_agent_df.empty else dataset_agent_df)
cd.plot_teacher_dataset_balance(dataset_agent_df, show=SHOW_FIGURES, save=SAVE_FIGURES)

## Teacher Dataset Cross-Seed Comparison

This tells you whether the teacher seeds are teaching the same sparse behavior or if each seed specializes different agents.

In [ ]:
if not dataset_agent_df.empty:
    pivot = dataset_agent_df.pivot_table(
        index="dataset",
        columns="agent",
        values=["teacher_action0_frac", "teacher_nonidle_frac", "was_overwritten_frac", "force_noop_frac"],
        aggfunc="mean",
    )
    display(pivot)

    import plotly.express as px
    fig = px.scatter(
        dataset_agent_df,
        x="teacher_nonidle_frac",
        y="was_overwritten_frac",
        color="dataset",
        symbol="agent",
        hover_data=["teacher_action0_frac", "force_noop_frac", "policy_nonidle_frac", "n"],
        labels={
            "teacher_nonidle_frac": "teacher non-idle fraction",
            "was_overwritten_frac": "heuristic overwrite fraction",
        },
        title="Teacher dataset: non-idle target vs heuristic overwrite rate",
    )
    fig.update_layout(template="plotly_white", height=650, width=1050)
    cd.save_fig(fig, "teacher_student_dataset_nonidle_vs_overwrite", save=SAVE_FIGURES)
    if SHOW_FIGURES:
        fig.show()

## Student BC Checkpoint Metrics

In [ ]:
student_ckpt_df, student_epoch_df = cd.load_teacher_student_checkpoints(STUDENT_CHECKPOINT_DIR)
print("student checkpoints:", len(student_ckpt_df))
display(student_ckpt_df.sort_values("checkpoint") if not student_ckpt_df.empty else student_ckpt_df)
print("epoch metrics:")
display(student_epoch_df.sort_values(["checkpoint", "epoch", "phase", "agent"]) if not student_epoch_df.empty else student_epoch_df)
cd.plot_student_bc_metrics(student_epoch_df, show=SHOW_FIGURES, save=SAVE_FIGURES)

## Student Prediction vs Teacher Target

This is the main sanity check after BC: does the student predict non-idle actions at the same rate as the teacher dataset, and is it missing rare non-idle actions?

In [ ]:
if not student_epoch_df.empty and not dataset_agent_df.empty:
    latest_eval = (
        student_epoch_df[student_epoch_df["phase"].eq("eval")]
        .sort_values(["checkpoint", "agent", "epoch"])
        .groupby(["checkpoint", "agent"], dropna=False, observed=True)
        .tail(1)
        .copy()
    )
    latest_eval = latest_eval.merge(
        student_ckpt_df[["checkpoint", "dataset", "balanced_nonidle_frac", "nonidle_weight", "aux_weight"]],
        on="checkpoint",
        how="left",
    )
    target = dataset_agent_df[["dataset", "agent", "teacher_nonidle_frac", "teacher_action0_frac", "was_overwritten_frac"]]
    comparison = latest_eval.merge(target, on=["dataset", "agent"], how="left")
    comparison["pred_minus_teacher_nonidle"] = comparison["pred_nonidle_frac"] - comparison["teacher_nonidle_frac"]
    display(comparison[[
        "checkpoint", "dataset", "agent", "epoch", "accuracy", "pred_nonidle_frac",
        "teacher_nonidle_frac", "pred_minus_teacher_nonidle", "false_noop_rate",
        "false_intervention_rate", "balanced_nonidle_frac", "nonidle_weight", "aux_weight",
    ]].sort_values(["checkpoint", "agent"]))

    import plotly.express as px
    fig = px.scatter(
        comparison,
        x="teacher_nonidle_frac",
        y="pred_nonidle_frac",
        color="checkpoint",
        symbol="agent",
        hover_data=["accuracy", "false_noop_rate", "false_intervention_rate", "pred_minus_teacher_nonidle"],
        title="Student predicted non-idle rate vs teacher dataset non-idle target",
        labels={"teacher_nonidle_frac": "teacher non-idle fraction", "pred_nonidle_frac": "student predicted non-idle fraction"},
    )
    fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line={"dash": "dash", "color": "black"})
    fig.update_layout(template="plotly_white", height=650, width=1050)
    cd.save_fig(fig, "teacher_student_pred_nonidle_vs_teacher_target", save=SAVE_FIGURES)
    if SHOW_FIGURES:
        fig.show()

    fig2 = px.bar(
        comparison,
        x="checkpoint",
        y="pred_minus_teacher_nonidle",
        color="agent",
        barmode="group",
        hover_data=["dataset", "teacher_nonidle_frac", "pred_nonidle_frac"],
        title="Student non-idle bias relative to teacher target",
    )
    fig2.update_layout(template="plotly_white", height=620, width=1250)
    fig2.update_xaxes(tickangle=-30)
    cd.save_fig(fig2, "teacher_student_nonidle_bias_by_agent", save=SAVE_FIGURES)
    if SHOW_FIGURES:
        fig2.show()
else:
    print("Need both student checkpoint metrics and dataset summaries for this comparison.")

## Standalone Full-Test Eval Of Student Checkpoints

Run full-test evals with `--eval-action-heuristic none`, then this cell will pick up the JSON summaries.

In [ ]:
full_test = cd.load_full_test_results()
if not full_test.empty:
    student_full_test = full_test[
        full_test["checkpoint"].astype(str).str.contains("teacher_student", case=False, na=False)
        | full_test["run_like"].astype(str).str.contains("bc|student|teacher", case=False, na=False)
    ].copy()
else:
    student_full_test = full_test

display(student_full_test.sort_values("survival_percent", ascending=False) if not student_full_test.empty else student_full_test)
cd.plot_full_test_results(student_full_test, show=SHOW_FIGURES, save=SAVE_FIGURES)

## Export Teacher-Student Tables

In [ ]:
out_dir = cd.FIG_DIR
out_dir.mkdir(parents=True, exist_ok=True)
if not dataset_df.empty:
    dataset_df.to_csv(out_dir / "teacher_student_dataset_summary.csv", index=False)
if not dataset_agent_df.empty:
    dataset_agent_df.to_csv(out_dir / "teacher_student_dataset_agent_summary.csv", index=False)
if not student_ckpt_df.empty:
    student_ckpt_df.to_csv(out_dir / "teacher_student_checkpoint_summary.csv", index=False)
if not student_epoch_df.empty:
    student_epoch_df.to_csv(out_dir / "teacher_student_epoch_metrics.csv", index=False)
if not student_full_test.empty:
    student_full_test.to_csv(out_dir / "teacher_student_full_test_results.csv", index=False)
print("Wrote CSV summaries to", out_dir)